## 01. Execution Result 기술통계

DA-00에서 생성한 Master Transaction Sample 73,410건을 이용하여
Ethereum Transaction이 블록에 기록된 이후 실제 Execution 결과가 어떻게 나타나는지 확인한다.

`receipt_status`는 다음과 같이 해석한다.

- `1` : Execution 성공
- `0` : Execution 실패

먼저 전체 및 Transaction Type별 성공·실패 건수와 실패율을 확인한다.

In [1]:
# 01. Master Sample 불러오기

import pandas as pd

DATA_PATH = (
    r"C:\Users\user\Desktop\26.09.01_project\ADP\ADP-DA"
    r"\03_digital_asset\data\processed"
    r"\da_master_transaction_sample_73410.csv"
)

df_master = pd.read_csv(DATA_PATH)

print(df_master.shape)

(73410, 13)


In [2]:
# 02. Receipt Status 정리

df_master["receipt_status"] = pd.to_numeric(
    df_master["receipt_status"],
    errors="coerce"
).astype("Int64")

status_by_type = (
    df_master
    .groupby(["transaction_type", "receipt_status"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "실패",
        1: "성공"
    })
)

status_by_type["전체"] = status_by_type.sum(axis=1)

status_by_type["실패율_pct"] = (
    status_by_type["실패"]
    / status_by_type["전체"]
    * 100
)

status_by_type["성공률_pct"] = (
    status_by_type["성공"]
    / status_by_type["전체"]
    * 100
)

display(status_by_type)

receipt_status,실패,성공,전체,실패율_pct,성공률_pct
transaction_type,,,,,
0,120,7880,8000,1.500000,98.500000
1,82,7918,8000,1.025000,98.975000
2,582,40828,41410,1.405458,98.594542
3,2,7998,8000,0.025000,99.975000
4,383,7617,8000,4.787500,95.212500


### 기술통계 해석

Master Transaction Sample 73,410건의 `receipt_status`를 확인한 결과,
모든 Transaction Type에서 블록에 기록된 Transaction과 실제 Execution Result가 항상 일치하지는 않았다.

특히 Transaction Type별 실패율에 차이가 확인되었다.

- Type 0: **1.500%**
- Type 1: **1.025%**
- Type 2: **1.405%**
- Type 3: **0.025%**
- Type 4: **4.788%**

즉, Ethereum Transaction이 블록에 기록되었다는 사실만으로
해당 요청이 실제로 성공적으로 실행되었다고 판단할 수 없다.

특히 Type 4는 다른 Transaction Type보다 상대적으로 높은 실패율을 보였으며,
반대로 Type 3은 매우 낮은 실패율을 보였다.

이는 FPG에서 외부 실행 요청이 전송되었다는 사실과 실제 실행 성공 여부를
동일한 상태로 처리해서는 안 된다는 근거가 된다.

따라서 FPG Runtime에서는

**Transaction 기록 → External Execution Result 확인 → 최종 PASS 판단**

과정을 분리할 필요가 있다.

In [3]:
# 03. 모집단 비중을 반영한 가중 Execution 실패율

population_weight = {
    0: 0.14345667,
    1: 0.00292973,
    2: 0.84013664,
    3: 0.00588700,
    4: 0.00758996
}

type_failure_rate = (
    df_master
    .groupby("transaction_type")["receipt_status"]
    .apply(lambda x: (x == 0).mean())
)

weighted_failure_rate = sum(
    type_failure_rate[tx_type] * population_weight[tx_type]
    for tx_type in type_failure_rate.index
)

weighted_success_rate = 1 - weighted_failure_rate

print(f"가중 Execution 실패율: {weighted_failure_rate * 100:.4f}%")
print(f"가중 Execution 성공률: {weighted_success_rate * 100:.4f}%")

가중 Execution 실패율: 1.4354%
가중 Execution 성공률: 98.5646%


### 전체 Execution Result 해석

불비례 층화표본의 모집단 비중을 복원하여 전체 Execution Result를 추정한 결과,

- 가중 Execution 실패율: **1.4354%**
- 가중 Execution 성공률: **98.5646%**

로 나타났다.

즉, 현재 Ethereum 환경에서 Transaction이 블록에 기록되었더라도
약 **1.44%는 실제 Execution이 실패**하는 것으로 추정된다.

따라서 `Transaction 존재` 또는 `외부 요청 전송 완료`만으로
실제 실행 성공을 판단하는 것은 적절하지 않다.

FPG Runtime에서는 외부 실행 요청을 전달한 상태와
실제 Execution Result를 별도의 상태로 관리하고,
Receipt 등 실행결과가 확인된 이후 최종 상태를 판단할 필요가 있다.

이 결과는 DA-01의 핵심 명제인

**Transaction Record ≠ Execution Result**

를 직접적으로 뒷받침한다.

## 3. 거래 유형별 실행 실패율 차이 가설검정

### 검증 목적

거래 유형에 따라 실행 성공·실패 비율이 실제로 다른지 확인한다.

### 가설

**H0 (귀무가설)**  
거래 유형과 실행 결과는 서로 독립이다.  
즉, 거래 유형이 달라도 성공·실패 비율에는 차이가 없다.

**H1 (대립가설)**  
거래 유형과 실행 결과는 서로 독립이 아니다.  
즉, 거래 유형에 따라 성공·실패 비율이 다르다.

### 검증 방법

- 카이제곱 독립성 검정
- 효과크기: 크래머의 V

In [4]:
# 03. 거래 유형과 실행 결과의 독립성 검정

from scipy.stats import chi2_contingency
import numpy as np

# 성공/실패 건수만 사용
contingency = status_by_type[["실패", "성공"]]

chi2, p_value, dof, expected = chi2_contingency(contingency)

# 효과크기: 크래머의 V
n = contingency.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 /
    (
        n * min(
            contingency.shape[0] - 1,
            contingency.shape[1] - 1
        )
    )
)

# 기대빈도 조건 확인
expected_min = expected.min()
expected_under_5 = (expected < 5).sum()
expected_ratio_under_5 = expected_under_5 / expected.size * 100

print(f"카이제곱 통계량: {chi2:,.4f}")
print(f"자유도: {dof}")
print(f"p-value: {p_value:.4e}")
print(f"크래머의 V: {cramers_v:.4f}")
print(f"최소 기대빈도: {expected_min:,.2f}")
print(f"기대빈도 5 미만 비율: {expected_ratio_under_5:.2f}%")

카이제곱 통계량: 672.6848
자유도: 4
p-value: 2.8604e-144
크래머의 V: 0.0957
최소 기대빈도: 127.39
기대빈도 5 미만 비율: 0.00%


### 가설검정 결과 해석

거래 유형과 실행 결과의 관계를 카이제곱 독립성 검정으로 확인한 결과,

- 카이제곱 통계량: **672.6848**
- 자유도: **4**
- p-value: **2.86 × 10⁻¹⁴⁴**
- 크래머의 V: **0.0957**
- 최소 기대빈도: **127.39**
- 기대빈도 5 미만 Cell: **0%**

으로 나타났다.

모든 Cell의 기대빈도가 5 이상이므로 카이제곱 검정의 적용 조건을 충족한다.

p-value가 유의수준 0.05보다 매우 작으므로 **귀무가설(H0)을 기각**한다.

즉, 거래 유형과 실행 결과는 서로 독립이라고 보기 어려우며,
**Transaction Type에 따라 Execution 성공·실패 비율에 통계적으로 유의한 차이가 존재한다.**

다만 크래머의 V는 **0.0957**로 나타나,
거래 유형과 실행 결과 사이의 전체적인 연관성 크기는 크지 않은 편이다.

따라서 본 검정의 핵심은
**“특정 Transaction Type이 실패를 결정한다”는 것이 아니라,
Transaction이 기록되었다는 사실만으로 실제 Execution 성공을 판단할 수 없다는 점**에 있다.

앞서 모집단 비중을 복원한 결과에서도 현재 Ethereum 환경의 Execution 실패율은 약 **1.4354%**로 추정되었다.

따라서 FPG에서는 외부 Transaction 생성 또는 전송 상태를 최종 PASS로 처리하지 않고,
**실제 Execution Result를 별도로 확인한 뒤 최종 상태를 결정하는 구조가 필요하다.**

## 4. 거래 유형별 실패율 사후검증

전체 카이제곱 검정에서 Transaction Type과 Execution Result 사이에
통계적으로 유의한 관계가 확인되었다.

하지만 카이제곱 검정만으로는 **어느 Transaction Type 사이에서 실패율 차이가 발생하는지** 알 수 없다.

따라서 Type 0~4의 실패율을 두 유형씩 비교하는 **두 비율 검정**을 수행한다.

총 10개의 비교가 동시에 이루어지므로,
다중검정으로 인한 오류 증가를 방지하기 위해 **Bonferroni 보정**을 적용한다.

- 유의수준: 0.05
- 비교 수: 10개
- 보정 유의수준: 0.005

또한 p-value뿐 아니라 각 Type 간 **실패율 차이(%p)**를 함께 확인한다.

In [5]:
# 04. Transaction Type별 실패율 사후검증

from itertools import combinations
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd

posthoc_results = []

type_summary = status_by_type.copy()

type_pairs = list(combinations(type_summary.index, 2))
n_tests = len(type_pairs)

alpha = 0.05
bonferroni_alpha = alpha / n_tests

for type_a, type_b in type_pairs:

    fail_a = int(type_summary.loc[type_a, "실패"])
    total_a = int(type_summary.loc[type_a, "전체"])

    fail_b = int(type_summary.loc[type_b, "실패"])
    total_b = int(type_summary.loc[type_b, "전체"])

    count = [fail_a, fail_b]
    nobs = [total_a, total_b]

    z_stat, p_value = proportions_ztest(
        count=count,
        nobs=nobs
    )

    rate_a = fail_a / total_a
    rate_b = fail_b / total_b

    rate_diff_pp = (rate_a - rate_b) * 100

    adjusted_p = min(
        p_value * n_tests,
        1.0
    )

    posthoc_results.append({
        "비교": f"Type {type_a} vs Type {type_b}",
        "Type A 실패율(%)": rate_a * 100,
        "Type B 실패율(%)": rate_b * 100,
        "실패율 차이(%p)": rate_diff_pp,
        "Z 통계량": z_stat,
        "원 p-value": p_value,
        "Bonferroni 보정 p-value": adjusted_p,
        "유의 여부": adjusted_p < alpha
    })

posthoc_df = pd.DataFrame(posthoc_results)

display(
    posthoc_df.round({
        "Type A 실패율(%)": 4,
        "Type B 실패율(%)": 4,
        "실패율 차이(%p)": 4,
        "Z 통계량": 4,
        "원 p-value": 6,
        "Bonferroni 보정 p-value": 6
    })
)

print(f"비교 수: {n_tests}")
print(f"Bonferroni 보정 유의수준: {bonferroni_alpha:.4f}")

,비교,Type A 실패율(%),Type B 실패율(%),실패율 차이(%p),Z 통계량,원 p-value,Bonferroni 보정 p-value,유의 여부
0,Type 0 vs Type 1,1.5000,1.0250,0.4750,2.6907,0.007130,0.071300,False
1,Type 0 vs Type 2,1.5000,1.4055,0.0945,0.6541,0.513029,1.000000,False
2,Type 0 vs Type 3,1.5000,0.0250,1.4750,10.7242,0.000000,0.000000,True
3,Type 0 vs Type 4,1.5000,4.7875,-3.2875,-11.9154,0.000000,0.000000,True
4,Type 1 vs Type 2,1.0250,1.4055,-0.3805,-2.7056,0.006819,0.068189,False
5,Type 1 vs Type 3,1.0250,0.0250,1.0000,8.7517,0.000000,0.000000,True
6,Type 1 vs Type 4,1.0250,4.7875,-3.7625,-14.1659,0.000000,0.000000,True
7,Type 2 vs Type 3,1.4055,0.0250,1.3805,10.4591,0.000000,0.000000,True
8,Type 2 vs Type 4,1.4055,4.7875,-3.3820,-20.0123,0.000000,0.000000,True
9,Type 3 vs Type 4,0.0250,4.7875,-4.7625,-19.6555,0.000000,0.000000,True


비교 수: 10
Bonferroni 보정 유의수준: 0.0050


### 사후검증 결과 해석

거래 유형별 실패율을 두 유형씩 비교하고 Bonferroni 보정을 적용한 결과,
10개 비교 중 **7개 조합에서 통계적으로 유의한 차이**가 확인되었다.

유의한 차이가 확인되지 않은 조합은 다음과 같다.

- Type 0 vs Type 1
- Type 0 vs Type 2
- Type 1 vs Type 2

즉, Type 0·1·2의 실패율은 각각 약 1.0~1.5% 수준으로 나타났으며,
다중비교 보정 이후에는 서로 유의한 차이가 확인되지 않았다.

반면 Type 3과 Type 4가 포함된 모든 비교에서는 유의한 차이가 나타났다.

- Type 3 실패율: **0.025%**
- Type 4 실패율: **4.7875%**

특히 Type 3과 Type 4의 실패율 차이는 **4.7625%p**로 가장 크게 나타났다.

따라서 전체 카이제곱 검정에서 확인된 거래 유형별 차이는
모든 Transaction Type이 서로 다른 실패 특성을 보이기 때문이라기보다,
**Type 3의 매우 낮은 실패율과 Type 4의 상대적으로 높은 실패율이 주요하게 기여한 것으로 해석할 수 있다.**

그러나 DA-01의 핵심 목적은 특정 Transaction Type의 실패 원인을 설명하는 것이 아니다.

모든 주요 Transaction Type에서 실제 Execution 실패가 관측되었고,
전체 모집단 기준 가중 실패율도 **1.4354%**로 추정되었다.

따라서 핵심 결론은

**Transaction이 기록되었다는 사실과 실제 Execution 성공은 동일한 상태로 처리할 수 없다**

는 것이다.

FPG Runtime에서는 외부 Transaction을 생성·전송한 상태와
실제 Execution Result 확인 상태를 분리하여 관리할 필요가 있다.